In [ ]:
from jaxtyping import Float, Array, Key, Scalar
import jax
import jax.numpy as jnp
import jax.scipy as jsp
import jax.random as jr
from flax import nnx
import optax

import numpy as np
from tqdm import tqdm
from matplotlib import pyplot as plt
from corner import corner
from emcee import EnsembleSampler

In [ ]:
Parameters = Float[Array, "dim"]
Observation = Float[Array, "dim"]

rngs = nnx.Rngs(42)


DIM = 3
PRIOR_MEAN = jnp.zeros(DIM)
PRIOR_COV = 3 * jnp.eye(DIM)
NOISE_COV = jr.uniform(rngs.prior(), shape=(DIM, DIM))
NOISE_COV = NOISE_COV @ NOISE_COV.T  # Ensure positive-definite covariance

@jax.jit
def sample_joint(rng: Key) -> tuple[Parameters, Observation]:
    rng_x, rng_y = jr.split(rng)
    x = jr.multivariate_normal(rng_x, PRIOR_MEAN, PRIOR_COV)
    y = jr.multivariate_normal(rng_y, x, NOISE_COV)
    return x, y


@jax.jit
def log_posterior(x: Parameters, y: Observation) -> Scalar:
    log_prior = jsp.stats.multivariate_normal.logpdf(x, PRIOR_MEAN, PRIOR_COV)
    log_likelihood = jsp.stats.multivariate_normal.logpdf(y, x, NOISE_COV)
    return log_prior + log_likelihood  # type: ignore

In [ ]:
HIDDEN_DIM = 512
LEARNING_RATE = 1e-4
BATCH_SIZE = 512
TOTAL_EXAMPLES = 1024_000


class Flow(nnx.Module):
    def __init__(self, *, rngs: nnx.Rngs):
        self.mlp = nnx.Sequential(
            nnx.Linear(DIM + DIM + 1, HIDDEN_DIM, rngs=rngs),
            nnx.silu,
            nnx.Linear(HIDDEN_DIM, HIDDEN_DIM, rngs=rngs),
            nnx.silu,
            nnx.Linear(HIDDEN_DIM, HIDDEN_DIM, rngs=rngs),
            nnx.silu,
            nnx.Linear(HIDDEN_DIM, DIM, rngs=rngs),
        )

    @nnx.jit
    def __call__(self, x: Parameters, t: Scalar, y: Observation) -> Parameters:
        h = jnp.concat([x, y, t[..., None]], axis=-1)
        return self.mlp(h)

    @nnx.jit
    def ode_step(
        self, x: Parameters, t: Scalar, y: Observation, dt: float
    ) -> Parameters:
        k1 = self(x, t, y)
        k2 = self(x + k1 * dt / 2, t + dt / 2, y)
        k3 = self(x + k2 * dt / 2, t + dt / 2, y)
        k4 = self(x + k3 * dt, t + dt, y)
        x = x + (k1 + 2 * k2 + 2 * k3 + k4) * dt / 6
        return x


@jax.jit
def get_train_batch(rng: Key) -> tuple[Parameters, Scalar, Observation, Parameters]:
    def phi(t: Scalar, x1: Parameters, x0: Parameters) -> Parameters:
        return x1 * t + x0 * (1 - t)

    def train_sample(rng: Key) -> tuple[Parameters, Scalar, Observation, Parameters]:
        rng_xy, rng_x0, rng_t = jr.split(rng, 3)
        x1, y = sample_joint(rng_xy)
        x0 = jr.normal(rng_x0, x1.shape)
        t = jr.uniform(rng_t, minval=0.0, maxval=1.0)

        xt = x1 * t + x0 * (1 - t)
        dx = jax.jacobian(phi)(t, x1, x0)
        return xt, t, y, dx

    return jax.vmap(train_sample)(jr.split(rng, BATCH_SIZE))


@nnx.jit
def train_step(
    model: Flow,
    optimizer: nnx.Optimizer,
    batch: tuple[Parameters, Scalar, Observation, Parameters],
) -> Scalar:
    def loss_fn(model):
        xt, t, y, dx = batch
        return jnp.mean((model(xt, t, y) - dx) ** 2)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)
    return loss


flow = Flow(rngs=rngs)
optimizer = nnx.Optimizer(flow, optax.adamw(learning_rate=LEARNING_RATE))

for step in (pbar := tqdm(range(TOTAL_EXAMPLES // BATCH_SIZE), desc="Training")):
    batch = get_train_batch(rngs.batch())
    loss = train_step(flow, optimizer, batch)
    pbar.set_postfix(loss=loss.item())

In [ ]:
RUNS = 10
SAMPLES = 32 * 1024
DIFFUSIONSTEPS = 16
MCMCWALKERS = 32
MCMCDISCARD = 300


def sample_from_flow(y: Observation):
    t = jnp.zeros((SAMPLES,))
    x = jr.normal(rngs.eval(), (SAMPLES, DIM))
    y = jnp.broadcast_to(y, (SAMPLES, *y.shape))
    dt = 1.0 / DIFFUSIONSTEPS
    for _ in tqdm(range(DIFFUSIONSTEPS)):
        x = flow.ode_step(x, t, y, dt)
        t += dt
    return np.array(x)


def sample_from_mcmc(y: Observation, x_true: Parameters = jnp.zeros(DIM)):
    p0 = np.random.randn(MCMCWALKERS, DIM)
    sampler = EnsembleSampler(MCMCWALKERS, DIM, log_posterior, args=(y,))
    sampler.run_mcmc(p0, nsteps=SAMPLES // MCMCWALKERS + MCMCDISCARD, progress=True)
    x = sampler.get_chain(flat=True, discard=MCMCDISCARD)
    return x


for run in range(RUNS):
    x_true, y = sample_joint(rngs.eval())

    print("Running flow sampling...")
    generated_samples = sample_from_flow(y)
    print("Running MCMC...")
    mcmc_samples = sample_from_mcmc(y, x_true)
    print()

    param_names = [f"$x_{i}$" for i in range(DIM)]
    fig = corner(mcmc_samples, labels=param_names, truths=x_true, color="blue")
    fig = corner(generated_samples, color="red", fig=fig)
    plt.show()